# Comparación y unificación de grados de la UPV y la UV

Los CSV se han obtenido mediante un **webscraper** a partir de información pública de ambas universidades. Cada fila representa un grado y campus, con sus notas de corte de 2022 a 2026 y la media de esos cinco años.

La comparación usa coincidencia exacta del campo `Grado`, sin exigir que coincida el campus. Si el mismo nombre aparece en ambas universidades, se calcula una media conjunta usando todas las notas disponibles de 2022 a 2026. Los grados exclusivos conservan su media original.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().resolve()
# Admitir ejecución desde la raíz o desde notebooks/.
if not (BASE_DIR / 'data' / 'raw').is_dir():
    BASE_DIR = BASE_DIR.parent
YEARS = ['2022', '2023', '2024', '2025', '2026']
UPV_PATH = BASE_DIR / 'data' / 'raw' / 'grados_upv_notas_corte_2022_2026.csv'
UV_PATH = BASE_DIR / 'data' / 'raw' / 'grados_uv_notas_corte_2022_2026.csv'
OUTPUT_PATH = BASE_DIR / 'data' / 'processed' / 'grados_upv_uv_unificado.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

assert UPV_PATH.exists(), f'No se encuentra {UPV_PATH}'
assert UV_PATH.exists(), f'No se encuentra {UV_PATH}'
upv = pd.read_csv(UPV_PATH, encoding='utf-8-sig')
uv = pd.read_csv(UV_PATH, encoding='utf-8-sig')
upv['Universidad'] = 'UPV'
uv['Universidad'] = 'UV'
for frame in (upv, uv):
    frame[YEARS] = frame[YEARS].apply(pd.to_numeric, errors='coerce')
    frame['Media_2022_2026'] = pd.to_numeric(frame['Media_2022_2026'], errors='coerce')
print(f'UPV: {len(upv)} filas; {upv["Grado"].nunique()} grados distintos')
print(f'UV: {len(uv)} filas; {uv["Grado"].nunique()} grados distintos')

## Descripción de los datos

Las columnas `2022`–`2026` son las notas de corte de cada curso. `Media_2022_2026` es la media calculada para cada fila. También se muestran los campus y los valores faltantes.

In [ ]:
def resumen(frame):
    return pd.Series({
        'Filas': len(frame),
        'Grados distintos': frame['Grado'].nunique(),
        'Campus distintos': frame['Campus'].nunique(),
        'Notas faltantes': int(frame[YEARS].isna().sum().sum()),
        'Media mínima': frame['Media_2022_2026'].min(),
        'Media máxima': frame['Media_2022_2026'].max(),
    })
display(pd.DataFrame({'UPV': resumen(upv), 'UV': resumen(uv)}).T)
print('Campus UPV')
display(upv.groupby('Campus').size().rename('Filas').to_frame())
print('Campus UV')
display(uv.groupby('Campus').size().rename('Filas').to_frame())

## Comparación por nombre exacto

Se distinguen los grados comunes, los exclusivos de la UPV y los exclusivos de la UV. No se consideran equivalentes los nombres parecidos, abreviados o con diferente ortografía.

In [ ]:
upv_grados = set(upv['Grado'].dropna())
uv_grados = set(uv['Grado'].dropna())
grados_comunes = sorted(upv_grados & uv_grados)
grados_solo_upv = sorted(upv_grados - uv_grados)
grados_solo_uv = sorted(uv_grados - upv_grados)
comparacion = pd.DataFrame({'Grupo': ['Comunes', 'Solo UPV', 'Solo UV'], 'Nombres': [len(grados_comunes), len(grados_solo_upv), len(grados_solo_uv)]})
display(comparacion)
display(pd.DataFrame({'Grados comunes': grados_comunes}))

## Unificación y media conjunta

Se conservan todas las filas de los dos CSV y se añade la universidad. Para cada nombre común se calcula `Media_Conjunta_2022_2026` con todas las notas de ambas universidades. `Media_2022_2026_Final` usa la media conjunta cuando existe coincidencia y la media original en los grados exclusivos.

In [ ]:
datos = pd.concat([upv, uv], ignore_index=True)
datos['Coincidencia'] = np.where(datos['Grado'].isin(grados_comunes), 'Común', 'Exclusivo')
def media_conjunta(grupo):
    valores = grupo[YEARS].to_numpy(dtype=float).ravel()
    valores = valores[~np.isnan(valores)]
    return float(valores.mean()) if len(valores) else np.nan
medias = datos[datos['Grado'].isin(grados_comunes)].groupby('Grado').apply(media_conjunta)
datos['Media_Conjunta_2022_2026'] = datos['Grado'].map(medias).round(2)
datos['Media_2022_2026_Final'] = datos['Media_Conjunta_2022_2026'].fillna(datos['Media_2022_2026']).round(2)
columnas = ['Universidad', 'Grado', 'Campus', *YEARS, 'Media_2022_2026', 'Coincidencia', 'Media_Conjunta_2022_2026', 'Media_2022_2026_Final']
unificado = datos[columnas].sort_values(['Grado', 'Universidad', 'Campus']).reset_index(drop=True)
unificado.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', float_format='%.2f')
print(f'CSV generado: {OUTPUT_PATH}')
display(unificado.head())

In [ ]:
assert len(unificado) == len(upv) + len(uv)
assert unificado['Media_2022_2026_Final'].notna().all()
assert set(unificado.loc[unificado['Coincidencia'] == 'Común', 'Grado']) == set(grados_comunes)
print('Validaciones superadas correctamente.')
display(unificado[unificado['Coincidencia'] == 'Común'][['Grado', 'Universidad', 'Campus', 'Media_2022_2026', 'Media_Conjunta_2022_2026', 'Media_2022_2026_Final']].drop_duplicates())

## Visualización

Se comparan las distribuciones de las medias originales. El CSV generado es el resultado principal del notebook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
datos.boxplot(column='Media_2022_2026', by='Universidad', ax=axes[0], grid=False)
axes[0].set_title('Distribución de medias')
axes[0].set_xlabel('Universidad')
axes[0].set_ylabel('Media 2022-2026')
comunes = datos[datos['Grado'].isin(grados_comunes)].groupby(['Grado', 'Universidad'])['Media_2022_2026'].mean().unstack()
comunes.plot(kind='bar', ax=axes[1])
axes[1].set_title('Medias originales de grados comunes')
axes[1].set_xlabel('Grado')
axes[1].set_ylabel('Media 2022-2026')
axes[1].tick_params(axis='x', labelrotation=90)
plt.suptitle('')
plt.tight_layout()
plt.show()